### Reply Generation

Goal:
Generate a human-reviewable customer support response using:
- Customer ticket text
- Predicted ticket type
- Predicted support queue
- Predicted priority
- Predicted sentiment
- Retrieved policy context

In [1]:
import os
import sys
import pandas as pd
import joblib

In [2]:
sys.path.append("..")

In [3]:
type_model = joblib.load("../models/ticket_type_baseline.pkl")
queue_model = joblib.load("../models/ticket_queue_baseline.pkl")
priority_model = joblib.load("../models/ticket_priority_baseline.pkl")
sentiment_model = joblib.load("../models/sentiment_model.pkl")

In [4]:
print("All models loaded successfully.")

All models loaded successfully.


In [5]:
from src.rag_pipeline import retrieve_policy_context

In [6]:
sample_ticket = "My laptop arrived damaged and I want a refund."

policy_context = retrieve_policy_context(sample_ticket, top_k=2)

policy_context

C:\Users\Shivam\Documents\shivam\customer-support-copilot\notebooks\..\src\rag_pipeline.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[{'source': 'refund_policy.txt',
  'content': 'Refunds are not allowed when:\n- The product was damaged because of customer misuse.\n- The refund request is made after 30 days.\n- The customer cannot provide proof of purchase.\n- The item was purchased from an unauthorized seller.\n\nFor damaged products, customers may choose either a refund or replacement.\n\nRefund requests should be assigned to the Billing or Customer Support queue depending on the issue.\nHigh-value refund disputes should be marked as high priority.'},
 {'source': 'refund_policy.txt',
  'content': 'Refund Policy\n\nCustomers can request a refund within 30 days of purchase.\n\nRefunds are allowed when:\n- The product is damaged on arrival.\n- The wrong item was delivered.\n- The product does not match the description.\n- The customer was charged incorrectly.\n- The customer received a defective item.'}]

### Making one full prediction

In [7]:
ticket = "My laptop arrived damaged and I want a refund."

predicted_type = type_model.predict([ticket])[0]
predicted_queue = queue_model.predict([ticket])[0]
predicted_priority = priority_model.predict([ticket])[0]
predicted_sentiment = sentiment_model.predict([ticket])[0]

retrieved_policy = retrieve_policy_context(ticket, top_k=2)

print("Ticket:", ticket)
print("Type:", predicted_type)
print("Queue:", predicted_queue)
print("Priority:", predicted_priority)
print("Sentiment:", predicted_sentiment)
print("Policy:", retrieved_policy[0]["source"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ticket: My laptop arrived damaged and I want a refund.
Type: Incident
Queue: Technical Support
Priority: high
Sentiment: negative
Policy: refund_policy.txt


### Reply Generator Function

In [8]:
def generate_support_reply(
    ticket_text,
    predicted_type,
    predicted_queue,
    predicted_priority,
    predicted_sentiment,
    retrieved_policy
):
    """
    Generates a human-reviewable customer support reply using
    ML predictions and retrieved policy context.
    """

    policy_text = retrieved_policy[0]["content"] if retrieved_policy else "No relevant policy context found."
    policy_source = retrieved_policy[0]["source"] if retrieved_policy else "No policy source"

    # Sentiment-aware opening
    if str(predicted_sentiment).lower() in ["negative", "neg", "0"]:
        opening = "We’re sorry to hear about the issue you’re facing."
    elif str(predicted_sentiment).lower() in ["positive", "pos", "1"]:
        opening = "Thank you for reaching out to us."
    else:
        opening = "Thank you for contacting customer support."

    # Priority-aware line
    if str(predicted_priority).lower() in ["high", "urgent", "critical"]:
        priority_line = (
            "We understand this may be urgent, so your request should be reviewed with priority."
        )
    elif str(predicted_priority).lower() in ["medium"]:
        priority_line = (
            "Your request has been noted and should be handled by the appropriate support team."
        )
    else:
        priority_line = (
            "Your request has been recorded and will be reviewed by our support team."
        )

    reply = f"""
Dear Customer,

{opening}

Based on your message, this request appears to be related to {predicted_type} and should be handled by the {predicted_queue} team.

{priority_line}

Relevant policy reference:
{policy_text}

Recommended next step:
Please keep your order details, product information, screenshots, or proof of purchase available if required by the support team.

Best regards,
Customer Support Team
""".strip()

    return {
        "reply": reply,
        "policy_source": policy_source
    }

In [9]:
result = generate_support_reply(
    ticket_text=ticket,
    predicted_type=predicted_type,
    predicted_queue=predicted_queue,
    predicted_priority=predicted_priority,
    predicted_sentiment=predicted_sentiment,
    retrieved_policy=retrieved_policy
)

print(result["reply"])
print("\nPolicy Source:", result["policy_source"])

Dear Customer,

We’re sorry to hear about the issue you’re facing.

Based on your message, this request appears to be related to Incident and should be handled by the Technical Support team.

We understand this may be urgent, so your request should be reviewed with priority.

Relevant policy reference:
Refunds are not allowed when:
- The product was damaged because of customer misuse.
- The refund request is made after 30 days.
- The customer cannot provide proof of purchase.
- The item was purchased from an unauthorized seller.

For damaged products, customers may choose either a refund or replacement.

Refund requests should be assigned to the Billing or Customer Support queue depending on the issue.
High-value refund disputes should be marked as high priority.

Recommended next step:
Please keep your order details, product information, screenshots, or proof of purchase available if required by the support team.

Best regards,
Customer Support Team

Policy Source: refund_policy.txt


### Full analysis Function

In [10]:
def analyze_ticket_and_generate_reply(ticket_text):
    """
    Runs ML predictions, retrieves policy context,
    and generates a customer support reply.
    """

    predicted_type = type_model.predict([ticket_text])[0]
    predicted_queue = queue_model.predict([ticket_text])[0]
    predicted_priority = priority_model.predict([ticket_text])[0]
    predicted_sentiment = sentiment_model.predict([ticket_text])[0]

    retrieved_policy = retrieve_policy_context(ticket_text, top_k=2)

    reply_result = generate_support_reply(
        ticket_text=ticket_text,
        predicted_type=predicted_type,
        predicted_queue=predicted_queue,
        predicted_priority=predicted_priority,
        predicted_sentiment=predicted_sentiment,
        retrieved_policy=retrieved_policy
    )

    return {
        "ticket": ticket_text,
        "predicted_type": predicted_type,
        "predicted_queue": predicted_queue,
        "predicted_priority": predicted_priority,
        "predicted_sentiment": predicted_sentiment,
        "policy_source": reply_result["policy_source"],
        "generated_reply": reply_result["reply"]
    }

In [11]:
output = analyze_ticket_and_generate_reply(
    "My charger stopped working after 6 months and I need warranty support."
)

output

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{'ticket': 'My charger stopped working after 6 months and I need warranty support.',
 'predicted_type': 'Incident',
 'predicted_queue': 'IT Support',
 'predicted_priority': 'high',
 'predicted_sentiment': 'negative',
 'policy_source': 'warranty_policy.txt',
 'generated_reply': 'Dear Customer,\n\nWe’re sorry to hear about the issue you’re facing.\n\nBased on your message, this request appears to be related to Incident and should be handled by the IT Support team.\n\nWe understand this may be urgent, so your request should be reviewed with priority.\n\nRelevant policy reference:\nEligible customers may receive repair, replacement, or service support.\n\nWarranty-related tickets should be assigned to Technical Support or Warranty Support.\n\nRecommended next step:\nPlease keep your order details, product information, screenshots, or proof of purchase available if required by the support team.\n\nBest regards,\nCustomer Support Team'}

In [12]:
test_tickets = [
    "My laptop arrived damaged and I want a refund.",
    "My order has not arrived and tracking is not updating.",
    "My phone battery drains very quickly.",
    "I cannot login to my account.",
    "My charger stopped working after 6 months and I need warranty support."
]

generated_results = []

for ticket in test_tickets:
    result = analyze_ticket_and_generate_reply(ticket)
    generated_results.append(result)

generated_df = pd.DataFrame(generated_results)
generated_df

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,ticket,predicted_type,predicted_queue,predicted_priority,predicted_sentiment,policy_source,generated_reply
0,My laptop arrived damaged and I want a refund.,Incident,Technical Support,high,negative,refund_policy.txt,"Dear Customer,\n\nWe’re sorry to hear about th..."
1,My order has not arrived and tracking is not u...,Problem,Technical Support,medium,negative,shipping_policy.txt,"Dear Customer,\n\nWe’re sorry to hear about th..."
2,My phone battery drains very quickly.,Incident,Customer Service,high,positive,technical_support_policy.txt,"Dear Customer,\n\nThank you for reaching out t..."
3,I cannot login to my account.,Incident,Technical Support,high,negative,account_policy.txt,"Dear Customer,\n\nWe’re sorry to hear about th..."
4,My charger stopped working after 6 months and ...,Incident,IT Support,high,negative,warranty_policy.txt,"Dear Customer,\n\nWe’re sorry to hear about th..."


### Inspecting Generated Replies

In [13]:
for i, row in generated_df.iterrows():
    print("=" * 100)
    print("Ticket:", row["ticket"])
    print("Type:", row["predicted_type"])
    print("Queue:", row["predicted_queue"])
    print("Priority:", row["predicted_priority"])
    print("Sentiment:", row["predicted_sentiment"])
    print("Policy Source:", row["policy_source"])
    print("\nGenerated Reply:\n")
    print(row["generated_reply"])

Ticket: My laptop arrived damaged and I want a refund.
Type: Incident
Queue: Technical Support
Priority: high
Sentiment: negative
Policy Source: refund_policy.txt

Generated Reply:

Dear Customer,

We’re sorry to hear about the issue you’re facing.

Based on your message, this request appears to be related to Incident and should be handled by the Technical Support team.

We understand this may be urgent, so your request should be reviewed with priority.

Relevant policy reference:
Refunds are not allowed when:
- The product was damaged because of customer misuse.
- The refund request is made after 30 days.
- The customer cannot provide proof of purchase.
- The item was purchased from an unauthorized seller.

For damaged products, customers may choose either a refund or replacement.

Refund requests should be assigned to the Billing or Customer Support queue depending on the issue.
High-value refund disputes should be marked as high priority.

Recommended next step:
Please keep your ord

In [14]:
REPORT_PATH = "../reports/generated_reply_examples.csv"

generated_df.to_csv(REPORT_PATH, index=False)

print("Saved generated replies to:", REPORT_PATH)

Saved generated replies to: ../reports/generated_reply_examples.csv
